# Setup & Initialization

In [ ]:
import random
import pyspark
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql import SparkSession, functions as F
from datetime import datetime
import ConnectionConfig as cc
from importlib import reload

reload(cc)
from pyspark.sql.functions import *

print(cc.__file__)

cc.setupEnvironment()
cc.listEnvironment()

In [ ]:
spark = cc.startLocalCluster("dimUserIncremental")
spark.getActiveSession()

## Reading the source table

In [3]:
cc.set_connectionProfile("catchem")

print(cc.create_jdbc())

jdbc:postgresql://postgres-data4:5432/catchem?user=postgres&password=data&ssl=false


In [4]:
# Load DataFrames
df_operational_user = cc.read_postgres_without_partition(spark, "user_table")
df_operational_treasure = cc.read_postgres_without_partition(spark, "treasure")
df_operational_treasure_log = cc.read_postgres_by_date(spark, "treasure_log", "log_time")
df_operational_city = cc.read_postgres_without_partition(spark, "city")
df_operational_country = cc.read_postgres_without_partition(spark, "country")

# Transforming the source to the dimension format

### Selecting and renaming columns for easier joins and transformations; also, we can filter or aggregate here if needed to optimize later steps

In [5]:
df_operational_city = df_operational_city.select(
    "city_id",
    "city_name",
    F.col("country_code").alias("country_id")
)

df_operational_country = df_operational_country.select(
    F.col("code").alias("country_id"),
    F.col("name").alias("country_name")
)

df_operational_user = df_operational_user.select(
    F.col("id").alias("user_id"),
    F.col("city_city_id").alias("city_id")
)

df_operational_treasure_log = df_operational_treasure_log \
    .groupBy(F.col("hunter_id").alias("user_id")) \
    .agg(F.count("*").alias("treasure_found"))

df_operational_treasure = df_operational_treasure \
    .select(F.col("owner_id").alias("user_id")) \
    .distinct()

print('user size: ', df_operational_user.count())
print('treasure size: ', df_operational_treasure.count())
print('treasure_log size: ', df_operational_treasure_log.count())
print('city size: ', df_operational_city.count())
print('country size: ', df_operational_country.count())

user size:  486523
treasure size:  33830
treasure_log size:  486340
city size:  1264588
country size:  249


In [6]:
# Join location data (Inner join usually fine for City -> Country)
df_location = df_operational_city.join(df_operational_country, "country_id")

# Prepare the "Dedicator" flag (Avoid distinct here to save performance; join handles it)
df_is_dedicator = df_operational_treasure.withColumn("dedicator", F.lit(True))

# Main Transformation
df_transformed = df_operational_user \
    .join(df_location, "city_id", "left") \
    .join(df_operational_treasure_log, "user_id", "left") \
    .join(df_is_dedicator, "user_id", "left") \
    .select(
    "user_id",
    "city_name",
    "country_name",
    # Use coalesce to handle nulls for users with no logs/flags
    F.coalesce(F.col("treasure_found"), F.lit(0)).cast("integer").alias("treasure_found"),
    F.coalesce(F.col("dedicator"), F.lit(False)).alias("dedicator")
) \
    .withColumn("Experience_level",
                F.when(F.col("treasure_found") == 0, "Starter")
                .when(F.col("treasure_found") < 4, "Amateur")
                .when(F.col("treasure_found") <= 10, "Professional")
                .otherwise("Pirate")
                ) \
    .withColumn("scd_start", F.current_timestamp()) \
    .withColumn("scd_end", F.lit(None).cast("timestamp")) \
    .withColumn("current", F.lit(True))

df_transformed = df_transformed.drop("treasure_found")

df_transformed.show(10, truncate=False)

+-------------------------------------------------+-------------------+------------------------+---------+----------------+--------------------------+-------+-------+
|user_id                                          |city_name          |country_name            |dedicator|Experience_level|scd_start                 |scd_end|current|
+-------------------------------------------------+-------------------+------------------------+---------+----------------+--------------------------+-------+-------+
|[00 02 66 4B E2 A6 4D 31 BE C6 3C 26 2B DA 05 B0]|Porbandar Mill Pura|India                   |false    |Amateur         |2026-03-19 12:52:41.845695|NULL   |true   |
|[00 05 0C 95 84 B6 45 82 80 D1 7F DC 34 C4 75 60]|Kinhupur           |India                   |false    |Amateur         |2026-03-19 12:52:41.845695|NULL   |true   |
|[00 07 66 C4 1B 47 49 59 98 B9 58 A9 66 C2 7A 8A]|Melaka             |Malaysia                |true     |Pirate          |2026-03-19 12:52:41.845695|NULL   |true   

# Change some user's data in the source to test incremental update

In [110]:
import uuid

# Get a random row using the DataFrame API
random_city_row = df_operational_city.select("city_id", "city_name").orderBy(F.rand()).limit(1).collect()

# Extract the value safely
# row[0] is the Row object, row[0][0] is the first column (city_id)
random_city_id = random_city_row[0][0] if random_city_row else 1

print(f"Random City ID selected: {str(uuid.UUID(bytes=bytes(random_city_id)))} - {random_city_row[0][1]}")

Random City ID selected: cda81b10-2160-4507-b470-e1e9d0b751c0 - Akram Abad


### Identify the specific user to update

In [112]:
target_user_row = df_operational_user.select("user_id").limit(1).collect()

target_user_id = target_user_row[0][0]

print(
    f"Random User ID selected: {str(uuid.UUID(bytes=bytes(target_user_id)))}")

Random User ID selected: 2e7ac8d1-00a6-4eec-943e-ccd2a012169e - Blanca Águilar


### Now we will update the city_id for this specific user to simulate a change in the source data that should trigger an update in our dimension table.

In [113]:
# This creates a "modified" version of existing user table
df_operational_user = df_operational_user.withColumn(
    "city_id",
    F.when(F.col("user_id") == target_user_id, F.lit(random_city_id)).otherwise(F.col("city_id"))
)
print(f"User Id: {uuid.UUID(bytes=bytes(target_user_id))}")
print(f"changes:  City ID -> {uuid.UUID(bytes=bytes(random_city_id))}")

User Id: 2e7ac8d1-00a6-4eec-943e-ccd2a012169e
changes:   Last Name -> 'UpdatedName', City ID -> cda81b10-2160-4507-b470-e1e9d0b751c0


### Next, let's also add a completely new user to test the insert logic for that part of our dimension as well

In [114]:
# Create a NEW user record
# Since user_id is a UUID (bytea), we use bytearray for the dummy ID
new_user_data = [(
    bytearray.fromhex('11111111222233334444555566667777'),  # A dummy UUID
    random_city_id
)]

# Important: Use the schema of df_operational_user so the union works perfectly
new_user_df = spark.createDataFrame(new_user_data, schema=df_operational_user.schema)

# Union the new record into the main DataFrame
df_operational_user = df_operational_user.union(new_user_df)
print(
    f"Added a new user with ID: 11111111-2222-3333-4444-555566667777 - city id: {uuid.UUID(bytes=bytes(new_user_data[0][1]))}")

Added a new user with ID: 11111111-2222-3333-4444-555566667777 - New User - city id: cda81b10-2160-4507-b470-e1e9d0b751c0


### Now let's update the count of treasures found for a random hunter to test the update logic for that part of our dimension as well

In [115]:
# Pick a random hunter_id from our current log DataFrame
target_hunter_row = (df_operational_treasure_log.select("user_id")
                     .filter(F.col("treasure_found") == 10)
                     .orderBy(F.rand()).limit(1).collect())

target_hunter_id = target_hunter_row[0][0]

print(f"Random Target Hunter ID  selected: {str(uuid.UUID(bytes=bytes(target_hunter_id)))}")

Random Target Hunter ID  selected: b195c6bf-e076-4708-ad20-3570e3f01e10


In [116]:
# Increment the count for that specific user
# F.col("treasure_found") + 1 adds one to the existing integer
df_operational_treasure_log = df_operational_treasure_log.withColumn(
    "treasure_found",
    F.when(F.col("user_id") == target_hunter_id, F.col("treasure_found") + 1)
    .otherwise(F.col("treasure_found"))
)

print(f"Incremented treasure count for user: {str(uuid.UUID(bytes=bytes(target_hunter_id)))}")

# Show the updated logs
df_operational_treasure_log.filter(F.col("user_id") == target_hunter_id).show(truncate=False)

Incremented treasure count for user: b195c6bf-e076-4708-ad20-3570e3f01e10
+-------------------------------------------------+--------------+
|user_id                                          |treasure_found|
+-------------------------------------------------+--------------+
|[B1 95 C6 BF E0 76 47 08 AD 20 35 70 E3 F0 1E 10]|11            |
+-------------------------------------------------+--------------+



### Finally, let's also add a new treasure owner to test the insert logic for that part of our dimension as well

In [117]:
# Perform a Left Anti Join
# This finds users in 'df_operational_user' who are NOT in 'df_operational_treasure'
df_non_owners = df_operational_user.join(
    df_operational_treasure,
    on="user_id",
    how="left_anti"
)

# Pick one random user_id from the non-owners
non_owner_row = df_non_owners.select("user_id").orderBy(F.rand()).limit(1).collect()

if non_owner_row:
    non_owner_id = non_owner_row[0][0]

    # Add this user to our treasure table (making them a "Dedicator")
    # We create a single-row DataFrame and union it
    new_owner_data = [(non_owner_id,)]
    new_owner_df = spark.createDataFrame(new_owner_data, schema=df_operational_treasure.schema)

    df_operational_treasure = df_operational_treasure.union(new_owner_df)

    print(f"Success! User {str(uuid.UUID(bytes=bytes(non_owner_id)))} is now a treasure owner.")
else:
    print("Optimization Note: Everyone already owns a treasure!")

# Verify the count increased
print(f"New total treasure owners: {df_operational_treasure.count()}")  ## 33831

Success! User eeb7f760-b0f9-43b0-b4e7-f3973d19c034 is now a treasure owner.
New total treasure owners: 33831


### With these changes in place, we have a mix of updates and inserts that will allow us to fully test our incremental load logic for the `dim_user` dimension. Now we can proceed with the transformation and incremental update steps.

In [7]:
# Join location data (Inner join usually fine for City -> Country)
df_location = df_operational_city.join(df_operational_country, "country_id")

# Prepare the "Dedicator" flag (Avoid distinct here to save performance; join handles it)
df_is_dedicator = df_operational_treasure.withColumn("dedicator", F.lit(True))

# Main Transformation
df_transformed = df_operational_user \
    .join(df_location, "city_id", "left") \
    .join(df_operational_treasure_log, "user_id", "left") \
    .join(df_is_dedicator, "user_id", "left") \
    .select(
    "user_id",
    "city_name",
    "country_name",
    F.coalesce(F.col("treasure_found"), F.lit(0)).cast("integer").alias("treasure_found"),
    F.coalesce(F.col("dedicator"), F.lit(False)).alias("dedicator")
) \
    .withColumn("experience_level",
                F.when(F.col("treasure_found") == 0, "Starter")
                .when(F.col("treasure_found") < 4, "Amateur")
                .when(F.col("treasure_found") <= 10, "Professional")
                .otherwise("Pirate")
                ) \
    .withColumn("scd_start", F.current_timestamp()) \
    .withColumn("scd_end", F.lit(None).cast("timestamp")) \
    .withColumn("current", F.lit(True))

df_transformed = df_transformed.drop("treasure_found")

df_transformed.show(10, truncate=False)

+-------------------------------------------------+-------------------+------------------------+---------+----------------+--------------------------+-------+-------+
|user_id                                          |city_name          |country_name            |dedicator|experience_level|scd_start                 |scd_end|current|
+-------------------------------------------------+-------------------+------------------------+---------+----------------+--------------------------+-------+-------+
|[00 02 66 4B E2 A6 4D 31 BE C6 3C 26 2B DA 05 B0]|Porbandar Mill Pura|India                   |false    |Amateur         |2026-03-19 12:54:43.952588|NULL   |true   |
|[00 05 0C 95 84 B6 45 82 80 D1 7F DC 34 C4 75 60]|Kinhupur           |India                   |false    |Amateur         |2026-03-19 12:54:43.952588|NULL   |true   |
|[00 07 66 C4 1B 47 49 59 98 B9 58 A9 66 C2 7A 8A]|Melaka             |Malaysia                |true     |Pirate          |2026-03-19 12:54:43.952588|NULL   |true   

# Incremental Update (SCD Type 2)

### Detect Changes

In [119]:
# Setup
run_timestamp = F.current_timestamp()
dt_dim_user = DeltaTable.forPath(spark, "spark-warehouse/dim_user")
df_current = dt_dim_user.toDF().filter(F.col("current") == True)

# Join to find changes
# We join the new transformed data against the current active records in DWH
detected_changes = df_transformed.alias("source").join(
    df_current.alias("dwh"),
    on="user_id",
    how="left"
).filter(
    (F.col("dwh.user_id").isNull()) |  # New Record
    (F.col("dwh.city_name") != F.col("source.city_name")) |
    (F.col("dwh.country_name") != F.col("source.country_name")) |
    (F.col("dwh.experience_level") != F.col("source.experience_level")) |
    (F.col("dwh.dedicator") != F.col("source.dedicator"))
).select(
    "source.*",
    F.col("dwh.user_id").alias("dwh_user_id"),
    F.col("dwh.city_name").alias("dwh_city_name"),
    F.col("dwh.country_name").alias("dwh_country_name"),
    F.col("dwh.dedicator").alias("dwh_dedicator"),
    F.col("dwh.experience_level").alias("dwh_experience_level"),
    F.col("dwh.scd_start").alias("dwh_scd_start")
)

print("Detected Changes:")
detected_changes.select("user_id", "city_name", "dwh_city_name", "experience_level",
                        "dwh_experience_level", "dedicator", "dwh_dedicator").show(truncate=False)

Detected Changes:
+-------------------------------------------------+-----------+-------------+--------------+--------------+----------------+--------------------+---------+-------------+
|user_id                                          |last_name  |dwh_last_name|city_name     |dwh_city_name |experience_level|dwh_experience_level|dedicator|dwh_dedicator|
+-------------------------------------------------+-----------+-------------+--------------+--------------+----------------+--------------------+---------+-------------+
|[EE B7 F7 60 B0 F9 43 B0 B4 E7 F3 97 3D 19 C0 34]|Parker     |Parker       |Khanewal G.P.O|Khanewal G.P.O|Amateur         |Amateur             |true     |false        |
|[B1 95 C6 BF E0 76 47 08 AD 20 35 70 E3 F0 1E 10]|Quigley    |Quigley      |Ellapuram     |Ellapuram     |Pirate          |Professional        |true     |true         |
|[11 11 11 11 22 22 33 33 44 44 55 55 66 66 77 77]|User       |NULL         |Akram Abad    |NULL          |Starter         |NULL    

### Prepare Upserts (The Union)

In [120]:
# Create the Insert records (New version of data)
df_inserts = detected_changes.select(
    "user_id", "city_name", "country_name",
    "dedicator", "experience_level"
).withColumn("scd_start", run_timestamp) \
    .withColumn("scd_end", F.lit(None).cast("timestamp")) \
    .withColumn("current", F.lit(True))

# Create the Update records (To close existing active records)
df_updates = detected_changes.filter(F.col("dwh_user_id").isNotNull()).select(
    F.col("user_id"),
    F.col("dwh_city_name").alias("city_name"),
    F.col("dwh_country_name").alias("country_name"),
    F.col("dwh_dedicator").alias("dedicator"),
    F.col("dwh_experience_level").alias("experience_level"),
    F.col("dwh_scd_start").alias("scd_start")
).withColumn("scd_end", run_timestamp) \
    .withColumn("current", F.lit(False))

# Combine them
df_upserts = df_inserts.unionByName(df_updates)

print("Upsert Table (Ready for Merge):")

df_upserts.show(truncate=False)

Upsert Table (Ready for Merge):
+-------------------------------------------------+----------+-----------+--------------+------------+--------------+---------+----------------+--------------------------+--------------------------+-------+
|user_id                                          |first_name|last_name  |city_name     |country_name|treasure_found|dedicator|experience_level|scd_start                 |scd_end                   |current|
+-------------------------------------------------+----------+-----------+--------------+------------+--------------+---------+----------------+--------------------------+--------------------------+-------+
|[EE B7 F7 60 B0 F9 43 B0 B4 E7 F3 97 3D 19 C0 34]|Cordelia  |Parker     |Khanewal G.P.O|Pakistan    |2             |true     |Amateur         |2026-03-14 23:22:05.859224|NULL                      |true   |
|[B1 95 C6 BF E0 76 47 08 AD 20 35 70 E3 F0 1E 10]|Brett     |Quigley    |Ellapuram     |India       |11            |true     |Pirate       

### The Delta Merge

In [121]:
dt_dim_user.alias("target").merge(
    source=df_upserts.alias("updates"),
    condition="target.user_id = updates.user_id AND target.current = true AND updates.current = false"
).whenMatchedUpdate(set={
    "scd_end": "updates.scd_end",
    "current": "updates.current"
}).whenNotMatchedInsert(values={
    "user_id": "updates.user_id",
    "city_name": "updates.city_name",
    "country_name": "updates.country_name",
    "dedicator": "updates.dedicator",
    "experience_level": "updates.experience_level",
    "scd_start": "updates.scd_start",
    "scd_end": "updates.scd_end",
    "current": "updates.current"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### History

In [122]:
# To see the file-level history of changes
dt_dim_user.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

+-------+-----------------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation   |operationParameters                                                                                                                                                                                                              |
+-------+-----------------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|2      |2026-03-14 23:22:41.142|MERGE       |{predicate -> ["(((user_id#7096 = user_id#6942) AND current#7106) AND NOT current#7382)"], matchedPredicates -> [{"actionType":"update"

In [53]:
# To see the actual data evolution
spark.read.format("delta").load("spark-warehouse/dim_user").orderBy("user_id", "scd_start").show()

+-------+--------------------+----------+-----------+-------------------+--------------------+----------------+---------+--------------+--------------------+--------------------+-------+
|user_sk|             user_id|first_name|  last_name|          city_name|        country_name|Experience_level|dedicator|treasure_found|           scd_start|             scd_end|current|
+-------+--------------------+----------+-----------+-------------------+--------------------+----------------+---------+--------------+--------------------+--------------------+-------+
|      3|[00 00 1C BF A8 E...|     Yanis|      Andre|           Leyssard|              France|         Amateur|    false|             3|2026-03-14 22:05:...|2026-03-14 22:15:...|  false|
| 487751|[00 00 1C BF A8 E...|     Yanis|UpdatedName| São Pedro da Torre|            Portugal|         Amateur|    false|             3|2026-03-14 22:15:...|                NULL|   true|
|      7|[00 00 2B 0B 08 D...|    Jaeden|Cruickshank|         Sch

## Cleaning up the environment (stop Spark and reset variables)

In [123]:
spark.stop()

In [124]:
%reset